# 🧩 Mini-Lab: Building a CLI

**Module: Delivery: CLI, API, and Debugging Lab 2** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Build** a CLI using `argparse` with arguments, help text, and descriptions
2. **Understand** exit codes and when to use `0` (success) vs non-zero (failure)
3. **Distinguish** between `stdout` (results) and `stderr` (diagnostics) and use each correctly
4. **Apply** layered configuration so CLI args override environment variables which override defaults

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Command-Line Interface | A CLI contract defines how a command is used and how it behaves: its arguments and flags, what `--help` promises, which exit codes mean success or failure, and what goes to stdout versus stderr. |
| Layered Configuration | Defaults live in code, environments override them at deploy time, and secrets never go in Git, so you can change behavior safely without editing source. |


## Why CLI Conventions Matter

A CLI that follows Unix conventions is **composable** — it can be piped, scripted, and tested by other tools. The three pillars are:

| Pillar | Convention | Why |
|--------|-----------|-----|
| **Exit codes** | `0` = success, non-zero = failure | Scripts and CI check `$?` / `%ERRORLEVEL%` to decide what to do next |
| **stdout vs stderr** | Results → `stdout`, logs/errors → `stderr` | So `cmd | jq .` works — pipes only carry stdout |
| **Help text** | `--help` / `-h` prints usage and exits | Users discover arguments without reading source code |

## Setup

We'll write Python scripts to disk using `%%writefile` and then run them with `!python` (or `subprocess`). This simulates the real workflow: **write a `.py` file, run it from the terminal**.

No pip installs needed — `argparse`, `os`, `sys`, and `subprocess` are all in the standard library.

In [1]:
import subprocess, sys, os
from pathlib import Path

from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

def run(cmd, env=None):
    """Run a shell command and print stdout, stderr, and exit code."""
    merged_env = {**os.environ, **(env or {})}
    result = subprocess.run(
        cmd, shell=True, capture_output=True, text=True, env=merged_env
    )
    if result.stdout:
        print("[stdout]", result.stdout, end="")
    if result.stderr:
        print("[stderr]", result.stderr, end="")
    print(f"[exit code] {result.returncode}")
    return result

## Step 1 — A Minimal CLI with `argparse`

Every CLI starts the same way: create a parser, add arguments, and call `parse_args()`. `argparse` gives you `--help` for free.

In [2]:
%%writefile greet.py
"""A tiny CLI that greets the user."""
import argparse
import sys


def main():
    parser = argparse.ArgumentParser(
        description="Greet someone by name."
    )
    parser.add_argument("name", help="Name of the person to greet")
    parser.add_argument(
        "--loud", action="store_true", help="SHOUT the greeting"
    )

    args = parser.parse_args()

    greeting = f"Hello, {args.name}!"
    if args.loud:
        greeting = greeting.upper()

    # Results go to stdout
    print(greeting)


if __name__ == "__main__":
    main()

Writing greet.py


In [3]:
# Check the auto-generated help text
run(f"{sys.executable} greet.py --help")

[stdout] usage: greet.py [-h] [--loud] name

Greet someone by name.

positional arguments:
  name        Name of the person to greet

options:
  -h, --help  show this help message and exit
  --loud      SHOUT the greeting
[exit code] 0


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe greet.py --help', returncode=0, stdout='usage: greet.py [-h] [--loud] name\n\nGreet someone by name.\n\npositional arguments:\n  name        Name of the person to greet\n\noptions:\n  -h, --help  show this help message and exit\n  --loud      SHOUT the greeting\n', stderr='')

In [4]:
# Normal run — result on stdout, exit code 0
run(f"{sys.executable} greet.py Alice")

[stdout] Hello, Alice!
[exit code] 0


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe greet.py Alice', returncode=0, stdout='Hello, Alice!\n', stderr='')

**Key observations:**
- `argparse` generated `--help` automatically from our `description` and `help` strings.
- The greeting appeared on **stdout** and the exit code was **0** (success).
- If you forget the required `name` argument, `argparse` prints an error to **stderr** and exits with code **2**.

In [5]:
# Missing required argument → stderr + exit code 2
run(f"{sys.executable} greet.py")

[stderr] usage: greet.py [-h] [--loud] name
greet.py: error: the following arguments are required: name
[exit code] 2


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe greet.py', returncode=2, stdout='', stderr='usage: greet.py [-h] [--loud] name\ngreet.py: error: the following arguments are required: name\n')

## Step 2 — Exit Codes and stderr

Your own error handling should follow the same pattern:
- **Errors/diagnostics → `stderr`** (so they don't pollute piped output)
- **Non-zero exit code** signals failure to the calling script or CI

Let's build a slightly more realistic CLI: a word counter that reads a file.

In [6]:
%%writefile wordcount.py
"""Count words in a text file."""
import argparse
import sys
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(
        description="Count words in a text file."
    )
    parser.add_argument("file", help="Path to the text file")
    args = parser.parse_args()

    path = Path(args.file)

    # --- Error handling: stderr + non-zero exit ---
    if not path.exists():
        print(f"error: file not found: {path}", file=sys.stderr)
        sys.exit(1)

    text = path.read_text(encoding="utf-8")
    word_count = len(text.split())

    # Diagnostics → stderr (so they don't break pipes)
    print(f"Reading {path.name}...", file=sys.stderr)

    # Result → stdout (pipeable)
    print(word_count)


if __name__ == "__main__":
    main()

Writing wordcount.py


In [7]:
# Create a small test file
Path("sample.txt").write_text(
    "The quick brown fox jumps over the lazy dog", encoding="utf-8"
)

# Success: word count on stdout, log on stderr, exit 0
run(f"{sys.executable} wordcount.py sample.txt")

[stdout] 9
[stderr] Reading sample.txt...
[exit code] 0


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe wordcount.py sample.txt', returncode=0, stdout='9\n', stderr='Reading sample.txt...\n')

In [8]:
# Failure: error on stderr, exit code 1
run(f"{sys.executable} wordcount.py missing.txt")

[stderr] error: file not found: missing.txt
[exit code] 1


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe wordcount.py missing.txt', returncode=1, stdout='', stderr='error: file not found: missing.txt\n')

Notice the pattern:

| Situation | Stream | Exit Code |
|-----------|--------|-----------|
| Success — the word count | `stdout` | `0` |
| Progress log ("Reading...") | `stderr` | — |
| File not found | `stderr` | `1` |

If another script does `count=$(python wordcount.py data.txt)`, the variable `count` contains **only** the number — the log line stays on stderr and doesn't interfere.

## Step 3 — Layered Configuration

Real CLIs get settings from **multiple sources** with a clear priority:

```
CLI argument  (highest priority — explicit user intent)
      ↑
Env variable  (set once per session / in .env)
      ↑
Hardcoded default  (lowest priority — safe fallback)
```

This is **layered configuration**. Let's add it to our greeter.

In [9]:
%%writefile greet_v2.py
"""Greeter with layered configuration."""
import argparse
import os
import sys


# --- Defaults (lowest priority) ---
DEFAULT_GREETING = "Hello"


def resolve_greeting(cli_value: str | None) -> str:
    """Resolve greeting from CLI arg → env var → default."""
    if cli_value is not None:
        return cli_value                           # CLI wins
    env_value = os.environ.get("GREETING_WORD")
    if env_value is not None:
        return env_value                           # Env var second
    return DEFAULT_GREETING                        # Default last


def main():
    parser = argparse.ArgumentParser(
        description="Greet someone (supports layered config)."
    )
    parser.add_argument("name", help="Name of the person to greet")
    parser.add_argument(
        "--greeting",
        default=None,
        help="Greeting word (default: $GREETING_WORD or 'Hello')",
    )
    args = parser.parse_args()

    word = resolve_greeting(args.greeting)
    print(f"{word}, {args.name}!")


if __name__ == "__main__":
    main()

Writing greet_v2.py


In [10]:
# Layer 1: No CLI arg, no env var → uses default "Hello"
run(f"{sys.executable} greet_v2.py Alice")

[stdout] Hello, Alice!
[exit code] 0


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe greet_v2.py Alice', returncode=0, stdout='Hello, Alice!\n', stderr='')

In [11]:
# Layer 2: Env var set, no CLI arg → uses env var
run(
    f"{sys.executable} greet_v2.py Alice",
    env={"GREETING_WORD": "Howdy"},
)

[stdout]

 Howdy, Alice!
[exit code] 0


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe greet_v2.py Alice', returncode=0, stdout='Howdy, Alice!\n', stderr='')

In [12]:
# Layer 3: CLI arg provided → overrides everything
run(
    f"{sys.executable} greet_v2.py Alice --greeting Bonjour",
    env={"GREETING_WORD": "Howdy"},
)

[stdout] Bonjour, Alice!
[exit code] 0


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe greet_v2.py Alice --greeting Bonjour', returncode=0, stdout='Bonjour, Alice!\n', stderr='')

The three runs demonstrate the priority chain in action:

| Run | CLI arg | Env var | Result | Source used |
|-----|---------|---------|--------|-------------|
| 1 | — | — | Hello, Alice! | Default |
| 2 | — | `Howdy` | Howdy, Alice! | Env var |
| 3 | `Bonjour` | `Howdy` | Bonjour, Alice! | CLI arg (highest priority) |

## Cleanup

In [13]:
# Remove the script files we created
for f in ["greet.py", "greet_v2.py", "wordcount.py", "sample.txt"]:
    Path(f).unlink(missing_ok=True)

print("Cleaned up.")

Cleaned up.


## 🎯 Summary

### Key Takeaways

1. **`argparse` gives you a CLI contract** — positional args, optional flags, and auto-generated `--help` with almost no boilerplate
2. **Exit codes signal success or failure** — `0` means success; any non-zero value tells scripts, CI, and cron that something went wrong
3. **stdout is for results, stderr is for everything else** — errors, progress logs, and diagnostics go to `stderr` so piped output stays clean
4. **Layered configuration (default → env var → CLI arg)** — lets users set defaults once in their environment while still overriding per-invocation from the command line